# Classi fini: lettura empirica delle micro-distribuzioni

            Questo foglio mette a confronto il livello ufficiale SBS 0-9 con le micro-classi osservate in Business Demography. Le quote sotto sono descrittive e non trasformano automaticamente il valore aggiunto in stime fini.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Dati di partenza

In [ ]:
sbs = enrich_sbs(read_project_csv("data/processed/official_eurostat_sbs.csv"))
bd = enrich_business_demography(read_project_csv("data/processed/business_demography_distribution.csv"))

pd.DataFrame(
    [
        {
            "dataset": "SBS",
            "anni": f'{int(sbs["year"].min())}-{int(sbs["year"].max())}',
            "classi": ", ".join(SIZE_ORDER_OFFICIAL),
        },
        {
            "dataset": "Business Demography",
            "anni": f'{int(bd["year"].min())}-{int(bd["year"].max())}',
            "classi": ", ".join(SIZE_ORDER_BUSINESS),
        },
    ]
)

## Micro-classi osservate fino a 9 dipendenti

In [ ]:
bd_latest = latest_year_with_values(bd, metric_code="V11910")
micro = (
    bd.query("metric_code == 'V11910' and year == @bd_latest and size_label in @SIZE_ORDER_BUSINESS")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
micro["size_label"] = pd.Categorical(micro["size_label"], SIZE_ORDER_BUSINESS, ordered=True)
micro_share = add_share(micro, ["country_label"])

fig = px.bar(
    micro_share.sort_values(["country_label", "size_label"]),
    x="country_label",
    y="share_pct",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_BUSINESS},
    labels={"country_label": "Paese", "share_pct": "% imprese fino a 9 dipendenti", "size_label": "Micro-classe"},
    title=f"Composizione osservata delle imprese fino a 9 dipendenti - {bd_latest}",
)
add_source_note(fig, "Eurostat Business Demography")
fig

## Italia: imprese e occupati nelle micro-classi

In [ ]:
micro_metrics = (
    bd.query("metric_code in ['V11910', 'V16910'] and year == @bd_latest and size_label in @SIZE_ORDER_BUSINESS")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label", "metric_code"], observed=True, as_index=False)["value"]
    .sum()
)
micro_metrics_share = add_share(micro_metrics, ["country_label", "metric_code"])
micro_metrics_share["indicatore"] = micro_metrics_share["metric_code"].map(METRIC_LABELS)

fig = px.bar(
    micro_metrics_share.query("country_label == 'Italia'"),
    x="size_label",
    y="share_pct",
    color="indicatore",
    barmode="group",
    category_orders={"size_label": SIZE_ORDER_BUSINESS},
    labels={"size_label": "Micro-classe", "share_pct": "%", "indicatore": "Indicatore"},
    title=f"Italia: imprese e occupati nelle micro-classi - {bd_latest}",
)
add_source_note(fig, "Eurostat Business Demography")
fig

## Pesi descrittivi per aprire la classe 0-9

In [ ]:
(
    micro_share.pivot_table(
        index="country_label",
        columns="size_label",
        values="share_pct",
        aggfunc="sum",
        observed=True,
    )
    .reindex(columns=SIZE_ORDER_BUSINESS)
    .round(1)
    .sort_values(SIZE_ORDER_BUSINESS[0], ascending=False)
)

## Valore aggiunto ufficiale della classe 0-9

In [ ]:
sbs_latest = latest_year_with_values(sbs, metric_code="AV_MEUR", filters={"size_label": "0-9"})
official_micro_va = (
    sbs.query("metric_code == 'AV_MEUR' and size_label == '0-9' and year == @sbs_latest")
    .dropna(subset=["value"])
    .groupby("country_label", as_index=False)["value"]
    .sum()
    .sort_values("value", ascending=False)
)

fig = px.bar(
    official_micro_va,
    x="country_label",
    y="value",
    labels={"country_label": "Paese", "value": "Milioni di euro"},
    title=f"Valore aggiunto osservato SBS della classe 0-9 - {sbs_latest}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig